# Zadanie 2, część I: modelowanie języka (BERT / HerBERT i GPT-2 + adaptery PEFT)

**Punktacja (6 pkt)**

| Część | Zakres | Punkty |
|---|---|---|
| A | klasy BERT w bibliotece Transformers (`BertModel`, `BertForMaskedLM`, `BertForSequenceClassification`, `BertForTokenClassification`) | 0.25 |
| B | biblioteka PEFT: quicktour i przykłady z repozytorium | 0.25 |
| C | powtórzenie tutoriala "PEFT of BERT base model to predict medical diagnosis" | 0.25 |
| D | klasy GPT-2 (`GPT2Model`, `GPT2LMHeadModel`, `GPT2ForSequenceClassification`, `GPT2ForTokenClassification`) | 0.25 |
| 1 | strojenie HerBERT + LoRA: klasyfikacja tekstu, klasyfikacja tokenów, ocena na zbiorze testowym | 1.5 |
| 2 | interaktywna wizualizacja przestrzeni wektorowej przykładów testowych (reprezentacja `[CLS]`) | 1.0 |
| 3 | powtórzenie 1 i 2 dla GPT-2 + LoRA (reprezentacja ostatniego tokenu, maskowanie `-100`) | 1.0 |
| 4 | augmentacja danych za pomocą tokenu `[MASK]` (BERT) | 0.5 |
| 5 | augmentacja danych generatywnym GPT-2 (temperatura, top-k, top-p) | 1.0 |

**Dane:** zbiór z Zadania 1 (`anotacja/zbior_finalny.csv`, 550 zdań PolEmo 2.0 z naszą etykietą sentymentu `POZ/NEG/NEU/AMB` i modyfikatorami `WZM/OSL/ODW`). Dzięki temu mamy dwa zadania na jednych danych:

- **klasyfikacja tekstu:** etykieta sentymentu całego zdania (4 klasy),
- **klasyfikacja tokenów:** typ modyfikatora każdego słowa (`O`, `WZM`, `OSL`, `ODW`).

**Modele:** `allegro/herbert-base-cased` (BERT-base dla polskiego) i `sdadas/polish-gpt2-medium` (GPT-2 dla polskiego). Oba stroimy adapterem LoRA z biblioteki PEFT.

> Notebook wymaga GPU do wygodnej pracy (HerBERT-base ok. 125 mln parametrów, polish-gpt2-medium ok. 355 mln). Zbiór jest mały (550 zdań), więc pojedyncze strojenie trwa minuty, ale GPT-2 medium na CPU będzie wolny.

In [1]:
# Instalacja (odkomentuj, jeśli brakuje pakietów). Wymagane: transformers >= 4.46, peft >= 0.10
# !pip install -U transformers peft datasets accelerate scikit-learn plotly pandas torch

In [2]:
import gc
import inspect
import random
import re
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import torch
import torch.nn as nn
from datasets import Dataset, load_dataset
from IPython.display import display
from peft import LoraConfig, PeftModel, TaskType, get_peft_model
from sklearn.cluster import KMeans
from sklearn.manifold import TSNE
from sklearn.metrics import (accuracy_score, adjusted_rand_score, classification_report,
                             confusion_matrix, f1_score, normalized_mutual_info_score,
                             silhouette_score)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import normalize
from transformers import (AutoModelForMaskedLM, AutoTokenizer, BertConfig, BertForMaskedLM,
                          BertForSequenceClassification, BertForTokenClassification, BertModel,
                          DataCollatorForLanguageModeling, DataCollatorForTokenClassification,
                          DataCollatorWithPadding, GPT2Config, GPT2ForSequenceClassification,
                          GPT2ForTokenClassification, GPT2LMHeadModel, GPT2Model, Trainer,
                          TrainingArguments, set_seed)

SEED = 42
set_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

BERT_NAME = "allegro/herbert-base-cased"
GPT_NAME = "sdadas/polish-gpt2-medium"
MLM_NAME = BERT_NAME            # model z głowicą MLM do augmentacji (zadanie 4)
NAMES = {"bert": BERT_NAME, "gpt2": GPT_NAME}

DATA = Path("anotacja/zbior_finalny.csv")   # wynik Zadania 1
OUT = Path("lm")                             # checkpointy, adaptery, wykresy
OUT.mkdir(exist_ok=True)

MAX_LEN = 128
SENT_LABELS = ["POZ", "NEG", "NEU", "AMB"]
MOD_TYPES = ["WZM", "OSL", "ODW"]
TOK_LABELS = ["O"] + MOD_TYPES
label2id = {l: i for i, l in enumerate(SENT_LABELS)}
tag2id = {l: i for i, l in enumerate(TOK_LABELS)}
COLORS = {"POZ": "#2ca02c", "NEG": "#d62728", "NEU": "#7f7f7f", "AMB": "#ff7f0e"}

pd.set_option("display.max_colwidth", 140)
print("urządzenie:", DEVICE)

W1028 13:06:03.986000 19636 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


urządzenie: cpu


## 0. Teoria w pigułce

### BERT i HerBERT

- **BERT** to enkoder Transformera (stos warstw: self-attention + sieć feed-forward), uczony dwuzadaniowo: **MLM** (Masked Language Model: ok. 15% tokenów jest maskowanych i model przewiduje je patrząc na kontekst z obu stron) oraz **NSP** (Next Sentence Prediction). Wejście to suma trzech osadzeń: tokenów (WordPiece), pozycji i segmentów.
- Na początku sekwencji stoi token specjalny `[CLS]`. Po przejściu przez wszystkie warstwy jego wektor (`last_hidden_state[:, 0]`) agreguje informację o całej sekwencji, więc służy jako **reprezentacja całego tekstu**.
- **HerBERT** (`allegro/herbert-base-cased`) ma architekturę BERT-base (12 warstw, 768 wymiarów, 12 głowic), ale jest uczony na polskich korpusach, bez NSP (tylko MLM), z tokenizerem BPE. Znaczniki specjalne mają inne nazwy (`<s>` pełni rolę `[CLS]`, `</s>` rolę `[SEP]`, `<mask>` rolę `[MASK]`), ale `<s>` nadal stoi na pozycji 0, więc kod z `[:, 0]` działa bez zmian.
- W Transformers ta sama baza (`BertModel`) dostaje różne **głowice**:

| Klasa | Głowica | Wyjście |
|---|---|---|
| `BertModel` | brak (+ opcjonalny pooler: `Linear` + `tanh` na `[CLS]`) | `last_hidden_state` (B, T, 768), `pooler_output` (B, 768) |
| `BertForMaskedLM` | `Linear` + GELU + LayerNorm + dekoder o rozmiarze słownika | `logits` (B, T, V) |
| `BertForSequenceClassification` | dropout + `Linear(768, K)` na `pooler_output` | `logits` (B, K) |
| `BertForTokenClassification` | dropout + `Linear(768, K)` na każdym tokenie | `logits` (B, T, K) |

- **Klasyfikacja tokenów a podsłowa:** etykiety mamy dla słów, a model widzi podsłowa. Standardowo etykietę dostaje **pierwsze podsłowo** każdego słowa, a pozostałe podsłowa i tokeny specjalne dostają `-100`. Funkcja straty `CrossEntropyLoss` ignoruje pozycje z `-100` (`ignore_index=-100`), więc nie wpływają one ani na stratę, ani na metryki.

### GPT-2

- **GPT-2** to dekoder Transformera z **maską przyczynową**: token na pozycji $t$ widzi tylko tokeny $\le t$. Jest uczony autoregresyjnie, czyli maksymalizuje $\sum_t \log p(x_t \mid x_{<t})$.
- Konsekwencje dla klasyfikacji:
    - **pierwszy token nie widzi nic poza sobą**, więc nie nadaje się na reprezentację tekstu (inaczej niż `[CLS]`). Reprezentacją całości jest **ostatni token sekwencji**, bo jako jedyny widzi cały tekst,
    - `GPT2ForSequenceClassification` bierze wektor ostatniego tokenu **różnego od `pad_token_id`** i przepuszcza go przez `Linear(n_embd, K)` (warstwa `score`). Dlatego trzeba ustawić `pad_token_id` w konfiguracji, a padding musi być po prawej stronie,
    - GPT-2 nie ma tokenu paddingu, więc standardowo przyjmuje się `pad_token = eos_token`. Maskujemy go etykietą `-100`, żeby nie liczył się do straty,
    - w klasyfikacji tokenów (`GPT2ForTokenClassification`) każdy token widzi tylko **lewy kontekst**. Dla modyfikatorów to wada, bo np. słowo "nie" nabiera sensu dopiero przy wyrazie, który po nim następuje. Spodziewamy się więc gorszych wyników niż dla HerBERT-a.
- Klasy GPT-2: `GPT2Model` (sam stos), `GPT2LMHeadModel` (+ `lm_head` o wagach sprzężonych z osadzeniami), `GPT2ForSequenceClassification` (+ `score`), `GPT2ForTokenClassification` (+ `classifier`).

### PEFT i LoRA

**Parameter-Efficient Fine-Tuning** to rodzina metod, które zamrażają model bazowy i uczą tylko niewielką liczbę dodanych parametrów. Dzięki temu strojenie jest tańsze, a wynik (adapter) waży megabajty zamiast gigabajtów i można go podmieniać na tym samym modelu bazowym.

**LoRA (Low-Rank Adaptation)**: zamiast uczyć macierz wag $W \in \mathbb{R}^{d \times k}$, uczymy jej niskorzędową poprawkę:

$$W' = W + \frac{\alpha}{r} B A, \qquad A \in \mathbb{R}^{r \times k},\ B \in \mathbb{R}^{d \times r},\ r \ll \min(d, k)$$

- $A$ jest inicjalizowana losowo, $B$ zerami, więc na starcie model jest identyczny z bazowym,
- $r$ to **ranga** (pojemność adaptera), $\alpha$ to skala (poprawka jest mnożona przez $\alpha / r$), `lora_dropout` reguluje przeuczenie,
- `target_modules` wskazuje warstwy, które dostają adapter. Dla BERT-a typowo `query` i `value`, dla GPT-2 `c_attn` (połączone Q, K, V; warstwa jest typu `Conv1D`, więc potrzebne jest `fan_in_fan_out=True`),
- `task_type` (`SEQ_CLS`, `TOKEN_CLS`, `CAUSAL_LM`) sprawia, że PEFT automatycznie **odmraża i zapisuje głowicę** zadania (`classifier` / `score`), bo jest nowa i losowa, więc musi się uczyć w całości,
- liczba parametrów jednej poprawki to $r(d + k)$ zamiast $d \cdot k$. Dla $d = k = 768$ i $r = 16$ to 24 576 zamiast 589 824, czyli ok. 4%.

### Augmentacja danych tekstowych

Zbiór ma 550 zdań (klasa AMB ma 40), więc augmentacja ma sens. Warunkiem jest **zachowanie etykiety**:

- **BERT i `[MASK]`:** maskujemy losowe słowo i podmieniamy je słowem przewidzianym przez model. Ryzyko: MLM proponuje także **antonimy** (*dobry* ↔ *zły*), bo występują w tych samych kontekstach (widzieliśmy to w części II Zadania 1 dla Word2Vec). Etykieta może się odwrócić, więc potrzebujemy filtra jakości.
- **GPT-2 i generacja:** model dostrojony na zdaniach z prefiksem klasy generuje nowe zdania. Jakość i różnorodność zależą od strategii dekodowania:
    - **temperatura** $T$: $p_i = \mathrm{softmax}(z_i / T)$, niska $T$ zaostrza rozkład (powtarzalne, bezpieczne), wysoka spłaszcza (różnorodne, ryzyko nonsensów),
    - **top-k:** losujemy tylko spośród $k$ najbardziej prawdopodobnych tokenów,
    - **top-p (nucleus):** losujemy z najmniejszego zbioru tokenów, których łączne prawdopodobieństwo $\ge p$ (zbiór jest adaptacyjny),
    - **greedy / beam search:** deterministyczne, dają powtarzalne, często zapętlone zdania, więc słabo nadają się do augmentacji.
- **Filtr:** zachowujemy tylko te syntetyczne zdania, dla których klasyfikator z kroku 1 (trenowany wyłącznie na zbiorze treningowym) potwierdza zamierzoną etykietę z pewnością $\ge$ progu.
- Augmentujemy **wyłącznie zbiór treningowy**. Walidacja i test zostają oryginalne, inaczej wynik byłby zawyżony.

## A. Klasy BERT w bibliotece Transformers [0.25 pkt]

Zapoznajemy się z implementacją `BertModel`, `BertForMaskedLM`, `BertForSequenceClassification` i `BertForTokenClassification`. Żeby nie pobierać wag, budujemy **malutkie modele z losowymi wagami** (2 warstwy, 32 wymiary) i sprawdzamy kształty wyjść oraz moduły głowic.

In [3]:
tiny_bert = BertConfig(vocab_size=100, hidden_size=32, num_hidden_layers=2, num_attention_heads=2,
                       intermediate_size=64, max_position_embeddings=64, num_labels=3)
x = {"input_ids": torch.randint(5, 100, (2, 10)), "attention_mask": torch.ones(2, 10, dtype=torch.long)}

for cls in [BertModel, BertForMaskedLM, BertForSequenceClassification, BertForTokenClassification]:
    m = cls(tiny_bert).eval()
    with torch.no_grad():
        out = m(**x)
    shapes = {k: tuple(v.shape) for k, v in out.items() if torch.is_tensor(v)}
    print(f"{cls.__name__:34s} moduły: {[n for n, _ in m.named_children()]}")
    print(f"{'':34s} wyjścia: {shapes}")

BertModel                          moduły: ['embeddings', 'encoder', 'pooler']
                                   wyjścia: {'last_hidden_state': (2, 10, 32), 'pooler_output': (2, 32)}
BertForMaskedLM                    moduły: ['bert', 'cls']
                                   wyjścia: {'logits': (2, 10, 100)}
BertForSequenceClassification      moduły: ['bert', 'dropout', 'classifier']
                                   wyjścia: {'logits': (2, 3)}
BertForTokenClassification         moduły: ['bert', 'dropout', 'classifier']
                                   wyjścia: {'logits': (2, 10, 3)}


In [4]:
# Jak głowice korzystają z reprezentacji bazowej? Fragmenty kodu źródłowego (forward)
for cls in [BertForSequenceClassification, BertForTokenClassification]:
    src = inspect.getsource(cls.forward)
    keys = ("pooled_output", "sequence_output", "self.classifier", "self.dropout", "loss_fct", "ignore")
    print(f"--- {cls.__name__}")
    print("\n".join(l.rstrip() for l in src.splitlines() if any(k in l for k in keys)) or "(przejrzyj kod źródłowy ręcznie)")

--- BertForSequenceClassification
        pooled_output = outputs[1]
        pooled_output = self.dropout(pooled_output)
        logits = self.classifier(pooled_output)
                loss_fct = MSELoss()
                    loss = loss_fct(logits.squeeze(), labels.squeeze())
                    loss = loss_fct(logits, labels)
                loss_fct = CrossEntropyLoss()
                loss = loss_fct(logits.view(-1, self.num_labels), labels.view(-1))
                loss_fct = BCEWithLogitsLoss()
                loss = loss_fct(logits, labels)
--- BertForTokenClassification
        sequence_output = outputs[0]
        sequence_output = self.dropout(sequence_output)
        logits = self.classifier(sequence_output)
            loss_fct = CrossEntropyLoss()
            loss = loss_fct(logits.view(-1, self.num_labels), labels.view(-1))


### Analiza (A)

Uzupełnij na podstawie wyników powyżej i lektury kodu (`modeling_bert.py`):

* **`BertModel`:** zwraca `last_hidden_state` (B, T, H) oraz `pooler_output` (B, H). *Pooler* to `Linear` + `tanh` zastosowane do wektora `[CLS]`. *(uzupełnij)*
* **`BertForSequenceClassification`:** używa `pooler_output`, a nie surowego `[CLS]`. Dlatego w zadaniu 2 wyciągamy `[CLS]` z `hidden_states` samodzielnie. *(uzupełnij: co to zmienia?)*
* **`BertForTokenClassification`:** głowica działa na `last_hidden_state` każdego tokenu, a strata pomija pozycje `-100`. *(uzupełnij)*
* **`BertForMaskedLM`:** zwraca logity o rozmiarze słownika dla każdej pozycji; do augmentacji bierzemy te z pozycji `[MASK]`. *(uzupełnij)*

## B. Biblioteka PEFT: quicktour i przykłady [0.25 pkt]

Przechodzimy przepis z quicktour: **konfiguracja** (`LoraConfig`) → **opakowanie modelu** (`get_peft_model`) → **trening** → **zapis adaptera** (`save_pretrained`) → **wczytanie** (`PeftModel.from_pretrained`) → **wiele adapterów** (`add_adapter` / `set_adapter`) → **scalenie** (`merge_and_unload`).

Przykłady z repozytorium (https://github.com/huggingface/peft/tree/main/examples) do przejrzenia: katalogi dla klasyfikacji sekwencji, klasyfikacji tokenów i modelowania przyczynowego (dokładne nazwy sprawdź w repozytorium, bo układ katalogów się zmienia).

In [5]:
demo_cfg = LoraConfig(task_type=TaskType.SEQ_CLS, r=16, lora_alpha=32, lora_dropout=0.1,
                      target_modules=["query", "value"])
base = BertForSequenceClassification.from_pretrained(BERT_NAME, num_labels=len(SENT_LABELS))
peft_demo = get_peft_model(base, demo_cfg)     # UWAGA: modyfikuje `base` w miejscu
peft_demo.print_trainable_parameters()

trainable = [n for n, p in peft_demo.named_parameters() if p.requires_grad]
print(f"\nParametry trenowalne: {len(trainable)} tensorów, np.:")
print("\n".join(trainable[:4] + ["..."] + trainable[-4:]))

config.json:   0%|          | 0.00/472 [00:00<?, ?B/s]

C:\Python314\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\pauko\.cache\huggingface\hub\models--allegro--herbert-base-cased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
C:\Python314\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  654MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: allegro/herbert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.sso.sso_relationship.weight            | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.sso.sso_relationship.bias              | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those param

model.safetensors: reconstructing file:   0%|          |  0.00B /  654MB            

model.safetensors: downloading bytes:           |  0.00B            

trainable params: 592,900 || all params: 125,038,856 || trainable%: 0.4742

Parametry trenowalne: 50 tensorów, np.:
base_model.model.bert.encoder.layer.0.attention.self.query.lora_A.default.weight
base_model.model.bert.encoder.layer.0.attention.self.query.lora_B.default.weight
base_model.model.bert.encoder.layer.0.attention.self.value.lora_A.default.weight
base_model.model.bert.encoder.layer.0.attention.self.value.lora_B.default.weight
...
base_model.model.bert.encoder.layer.11.attention.self.value.lora_A.default.weight
base_model.model.bert.encoder.layer.11.attention.self.value.lora_B.default.weight
base_model.model.classifier.modules_to_save.default.weight
base_model.model.classifier.modules_to_save.default.bias


In [6]:
# zapis i wczytanie adaptera (zapisywane są tylko wagi LoRA + głowica)
path = OUT / "demo_adapter"
peft_demo.save_pretrained(path)
for f in sorted(path.iterdir()):
    print(f"{f.name:28s} {f.stat().st_size / 1024:9.1f} KB")

fresh = BertForSequenceClassification.from_pretrained(BERT_NAME, num_labels=len(SENT_LABELS))
loaded = PeftModel.from_pretrained(fresh, path)

# wiele adapterów na jednym modelu bazowym
loaded.add_adapter("drugi", demo_cfg)
loaded.set_adapter("drugi")
print("\naktywny adapter:", loaded.active_adapter, "| dostępne:", list(loaded.peft_config))
loaded.set_adapter("default")

# scalenie adaptera z wagami bazowymi (szybszy inferencja, brak zależności od PEFT)
merged = loaded.merge_and_unload()
print("po merge_and_unload:", type(merged).__name__)
del base, peft_demo, fresh, loaded, merged
gc.collect()

adapter_config.json                1.2 KB
adapter_model.safetensors       2322.8 KB
README.md                          5.1 KB


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: allegro/herbert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.sso.sso_relationship.weight            | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.sso.sso_relationship.bias              | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those param


aktywny adapter: drugi | dostępne: ['default', 'drugi']
po merge_and_unload: BertForSequenceClassification


6739

### Analiza (B)

* **Ile parametrów jest trenowanych w naszej konfiguracji (r=16, query+value) i jaki to ułamek modelu?** *(uzupełnij z `print_trainable_parameters`)*
* **Co wchodzi do pliku adaptera?** Wagi LoRA oraz nowa głowica klasyfikacyjna (dzięki `task_type`), a nie cały model. *(uzupełnij rozmiar)*
* **Kiedy przydają się wiele adapterów i `merge_and_unload`?** *(uzupełnij: jeden model bazowy, wiele zadań; scalanie na potrzeby wdrożenia)*

## C. Powtórzenie tutoriala: PEFT BERT-base do diagnozy medycznej [0.25 pkt]

Tutorial: *Parameter-Efficient Fine-Tuning (PEFT) of BERT base model to predict medical diagnosis* (N. Ahmed, Medium). Zbiór `gretelai/symptom_to_diagnosis`: opisy objawów (`input_text`) i 22 diagnozy (`output_text`), 853 przykłady treningowe i 212 testowych.

Powtarzamy tutorial **bez zmian**: baseline `bert-base-uncased` bez strojenia (pipeline), LoRA (`r=64, lora_alpha=1, lora_dropout=0.1`), `padding="max_length"`, `learning_rate=2e-3`, batch 4, 15 epok, zbiór testowy jako `eval_dataset` (z `load_best_model_at_end=True`), zapis adaptera, wczytanie i predykcja. Jedyne zmiany to nazwy argumentów wymagane przez nowsze wersje Transformers.

Uwaga: tutorial używa zbioru testowego także do wyboru najlepszego checkpointu, więc podany wynik (94.34%) jest nieco optymistyczny. Odtwarzamy to świadomie, a komentarz jest w analizie (C).

In [7]:
import os
from peft import PeftConfig
from transformers import AutoModelForSequenceClassification, pipeline


# używane też w dalszej części notebooka (run_training), więc poza blokiem RUN_TUTORIAL
def seq_metrics(p):
    preds = p.predictions.argmax(-1)
    y = p.label_ids
    return {"accuracy": accuracy_score(y, preds), "macro_f1": f1_score(y, preds, average="macro")}


RUN_TUTORIAL = True

if RUN_TUTORIAL:
    # --- dane ---
    data_files = {"train": "train.jsonl", "test": "test.jsonl"}
    dataset_t = load_dataset("gretelai/symptom_to_diagnosis", data_files=data_files)
    dataset_t = dataset_t.rename_column("output_text", "label")
    print(dataset_t)
    for entry in dataset_t["train"].select(range(5)):
        print("INPUT: {} \nOUTPUT: {}\n".format(entry["input_text"], entry["label"]))

    train_counts = pd.DataFrame({"Diagnosis": dataset_t["train"]["label"]}).groupby("Diagnosis").size().reset_index(
        name="train_set")
    test_counts = pd.DataFrame({"Diagnosis": dataset_t["test"]["label"]}).groupby("Diagnosis").size().reset_index(
        name="test_set")
    display(train_counts.merge(test_counts, on="Diagnosis"))

    # --- baseline: pretrained BERT z losową głowicą (w tutorialu 4.25%) ---
    sorted_labels_t = sorted(set(dataset_t["train"]["label"]))
    label2id_t = dict(zip(sorted_labels_t, range(len(sorted_labels_t))))
    id2label_t = dict(zip(range(len(sorted_labels_t)), sorted_labels_t))

    tokenizer_t = AutoTokenizer.from_pretrained("bert-base-uncased")
    foundation_model_t = AutoModelForSequenceClassification.from_pretrained(
        "bert-base-uncased", num_labels=len(label2id_t), label2id=label2id_t, id2label=id2label_t)

    classifier_t = pipeline("text-classification", model=foundation_model_t, tokenizer=tokenizer_t)
    predicted_labels_t = classifier_t(dataset_t["test"]["input_text"])
    test_array_t = np.asarray(dataset_t["test"]["label"])
    pred_array_t = np.asarray([item["label"] for item in predicted_labels_t])
    foundation_accuracy_t = round(sum(test_array_t == pred_array_t) * 100 / len(test_array_t), 2)
    print(f"Foundation Model Accuracy: {foundation_accuracy_t}%")  # w tutorialu: 4.25%

README.md:   0%|          | 0.00/2.46k [00:00<?, ?B/s]

C:\Python314\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\pauko\.cache\huggingface\hub\datasets--gretelai--symptom_to_diagnosis. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


train.jsonl:   0%|          | 0.00/171k [00:00<?, ?B/s]

test.jsonl:   0%|          | 0.00/42.5k [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

...

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
C:\Python314\Lib\site-

Training Loss,Validation Loss,Step,Accuracy,Macro F1
No log,3.165376,0,0.061321,0.010220


baseline bez strojenia: {'eval_loss': 3.1653764247894287, 'eval_accuracy': 0.06132075471698113, 'eval_macro_f1': 0.010219943548277132}


In [ ]:
if RUN_TUTORIAL:
    # --- konfiguracja PEFT-LoRA (jak w tutorialu) ---
    lora_config_t = LoraConfig(task_type=TaskType.SEQ_CLS, r=64, lora_alpha=1, lora_dropout=0.1)
    peft_model_t = get_peft_model(foundation_model_t, lora_config_t)
    print(peft_model_t.bert)
    peft_model_t.print_trainable_parameters()  # w tutorialu: 2,376,214 || 111,875,372 || 2.1240%
    print("moduły z adapterem:",
          sorted({n.split(".")[-4] for n, _ in peft_model_t.named_parameters() if "lora_A" in n}))


    # --- tokenizacja: padding="max_length", jak w tutorialu ---
    def preprocess_function_t(examples):
        tokens = tokenizer_t(examples["input_text"], padding="max_length", truncation=True)
        tokens["label"] = [label2id_t[l] for l in examples["label"]]
        return tokens


    tokenized_ds_t = {split: dataset_t[split].map(preprocess_function_t, batched=True) for split in ["train", "test"]}
    print(tokenized_ds_t)


    def compute_metrics_t(eval_pred):
        predictions, labels = eval_pred
        predictions = np.argmax(predictions, axis=1)
        return {"accuracy": (predictions == labels).mean() * 100}


    # --- trening: zbiór testowy jako eval_dataset, jak w tutorialu ---
    trainer_t = Trainer(
        model=peft_model_t,
        args=TrainingArguments(
            output_dir="bert-lora",
            learning_rate=2e-3,
            per_device_train_batch_size=4,
            per_device_eval_batch_size=4,
            num_train_epochs=15,
            weight_decay=0.01,
            eval_strategy="epoch",  # w tutorialu: evaluation_strategy
            save_strategy="epoch",
            load_best_model_at_end=True,
            report_to="none",
            label_names=["labels"],
        ),
        train_dataset=tokenized_ds_t["train"],
        eval_dataset=tokenized_ds_t["test"],
        processing_class=tokenizer_t,  # w tutorialu: tokenizer
        data_collator=DataCollatorWithPadding(tokenizer=tokenizer_t),
        compute_metrics=compute_metrics_t,
    )
    print("Starting to train...")
    trainer_t.train()
    print("Accuracy po strojeniu:", trainer_t.evaluate()["eval_accuracy"])  # w tutorialu: 94.34%

In [ ]:
if RUN_TUTORIAL:
    # --- zapis adaptera i jego rozmiar ---
    peft_bert_model_path = "fine-tuned-peft-model-weights/"
    peft_model_t.save_pretrained(peft_bert_model_path)
    for file_name in os.listdir(peft_bert_model_path):
        file_size = os.path.getsize(peft_bert_model_path + file_name)
        print(
            f"File Name: {file_name}; File Size: {file_size / 1024:.2f}KB")  # tutorial: adapter_model.safetensors ~9289KB

    # --- wczytanie zapisanego adaptera i predykcja ---
    config_t = PeftConfig.from_pretrained(peft_bert_model_path)
    model_t = BertForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=22)
    model_t = PeftModel.from_pretrained(model_t, peft_bert_model_path)

    trainer_pred_t = Trainer(model=model_t, args=TrainingArguments(output_dir="tmp_trainer", report_to="none"),
                             data_collator=DataCollatorWithPadding(tokenizer=tokenizer_t))
    test_predictions_t = trainer_pred_t.predict(tokenized_ds_t["test"])
    acc_loaded = (test_predictions_t.predictions.argmax(-1) == test_predictions_t.label_ids).mean() * 100
    print(f"Accuracy wczytanego adaptera: {acc_loaded:.2f}%")

trainable params: 2,376,214 || all params: 111,875,372 || trainable%: 2.1240
moduły z adapterem: ['lora_A']


C:\Python314\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1
1,No log,2.486815,0.255814,0.191471
2,No log,1.368682,0.558140,0.504367
3,2.241106,0.915210,0.651163,0.642582
4,2.241106,0.714314,0.825581,0.780408


C:\Python314\Lib\site-packages\peft\utils\other.py:1508: UserWarning: Unable to fetch remote file due to the following error [Errno 11001] getaddrinfo failed - silently ignoring the lookup for the file config.json in bert-base-uncased.
  warnings.warn(
C:\Python314\Lib\site-packages\peft\utils\save_and_load.py:313: UserWarning: Could not find a config file in bert-base-uncased - will assume that the vocabulary was not modified.
  warnings.warn(
C:\Python314\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
C:\Python314\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
C:\Python314\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no

### Analiza (C)

* **Wynik:** baseline *(uzupełnij)* → po strojeniu *(uzupełnij)*; oryginał: 4.25% → 94.34% accuracy. Czy wynik jest porównywalny? *(uzupełnij)* Baseline zależy od losowej inicjalizacji głowicy, więc przy 22 klasach oczekujemy ok. 4–5%, ale nie dokładnie 4.25%.
* **Liczba parametrów i rozmiar adaptera:** trenowane parametry *(uzupełnij; tutorial: 2 376 214, 2.12%)*, rozmiar `adapter_model.safetensors` *(uzupełnij; tutorial: ok. 9 MB)*.
* **Wyciek w tutorialu:** zbiór testowy jest jednocześnie `eval_dataset`, a `load_best_model_at_end=True` wybiera po nim najlepszy checkpoint. Podany wynik jest więc optymistyczny. W naszych własnych eksperymentach (zadania 1–6) używamy osobnej walidacji. *(uzupełnij)*
* **`lora_alpha=1` przy `r=64`:** skala poprawki to $\alpha / r = 1/64$, więc wpływ adaptera jest bardzo mały, a model nadrabia to wysokim `learning_rate=2e-3`. W naszych eksperymentach używamy typowego $\alpha = 2r$ i mniejszego LR. *(uzupełnij obserwacje)*
* **Które warstwy dostały adapter?** Tekst tutoriala mówi o "Query i Key", ale wypisana architektura (`print(peft_model_t.bert)`) pokazuje adaptery w `query` i `value`, a `key` pozostaje zwykłym `Linear`. Potwierdza to też wypis "moduły z adapterem". *(potwierdź)*

## D. Klasy GPT-2 w bibliotece Transformers [0.25 pkt]

Analogicznie jak w części A: małe modele z losowymi wagami, kształty wyjść i moduły. Dodatkowo sprawdzamy dwie własności ważne dla zadania 3:

1. `GPT2ForSequenceClassification` bierze **ostatni token niebędący paddingiem** (wynik dla sekwencji z paddingiem po prawej = wynik dla tej samej sekwencji bez paddingu),
2. w `GPT2LMHeadModel` wagi `lm_head` są **sprzężone** z osadzeniami tokenów.

In [ ]:
tiny_gpt = GPT2Config(vocab_size=100, n_embd=32, n_layer=2, n_head=2, n_positions=64,
                      num_labels=3, pad_token_id=0)
x = {"input_ids": torch.randint(1, 100, (2, 10)), "attention_mask": torch.ones(2, 10, dtype=torch.long)}

for cls in [GPT2Model, GPT2LMHeadModel, GPT2ForSequenceClassification, GPT2ForTokenClassification]:
    m = cls(tiny_gpt).eval()
    with torch.no_grad():
        out = m(**x)
    shapes = {k: tuple(v.shape) for k, v in out.items() if torch.is_tensor(v)}
    print(f"{cls.__name__:32s} moduły: {[n for n, _ in m.named_children()]}")
    print(f"{'':32s} wyjścia: {shapes}")

lm_demo = GPT2LMHeadModel(tiny_gpt)
print("\nlm_head sprzężony z wte:", lm_demo.lm_head.weight is lm_demo.transformer.wte.weight)

# ostatni token bez paddingu
m = GPT2ForSequenceClassification(tiny_gpt).eval()
ids = torch.randint(1, 100, (1, 6))
padded = torch.cat([ids, torch.zeros(1, 4, dtype=torch.long)], dim=1)
mask = torch.cat([torch.ones(1, 6), torch.zeros(1, 4)], dim=1).long()
with torch.no_grad():
    a = m(input_ids=ids).logits
    b = m(input_ids=padded, attention_mask=mask).logits
print("logity z paddingiem == logity bez paddingu:", torch.allclose(a, b, atol=1e-5))

### Analiza (D)

* **Czym różni się `GPT2ForSequenceClassification` od `BertForSequenceClassification`?** Brak pooler-a i `[CLS]`; reprezentacją jest ostatni token (jedyny widzący cały tekst dzięki masce przyczynowej), wybierany na podstawie `pad_token_id`. *(uzupełnij na podstawie kodu `modeling_gpt2.py`)*
* **Dlaczego trzeba ustawić `pad_token_id` w konfiguracji i padding po prawej?** *(uzupełnij)*
* **Czym grozi `pad_token = eos_token`?** Jeśli token końca tekstu pojawi się w samych danych, zostanie potraktowany jak padding. W naszych danych tokenizer nie dodaje `eos`, więc problem nie występuje. *(potwierdź)*

## Dane: zbiór z Zadania 1, podział na train / val / test

Wczytujemy `zbior_finalny.csv`. Dwa rodzaje etykiet:

- **tekst:** kolumna `sentyment` (po zmianie nazwy `label`),
- **tokeny:** kolumna `modyfikatory` w formacie `fraza=TYP; fraza=TYP` zamieniamy na etykiety słów `O/WZM/OSL/ODW` (funkcje z Zadania 1; fraza, której nie da się dopasować do tekstu, jest pomijana).

Teksty są już stokenizowane na słowa (`text.split()`), więc do tokenizera podajemy listy słów (`is_split_into_words=True`), a etykiety wyrównujemy do podsłów.

Podział warstwowany po etykiecie sentymentu: **70% train / 15% val / 15% test**. Walidacja służy do wyboru checkpointu, a test jest użyty **dopiero na końcu**. Zbiór testowy jest mały (ok. 83 zdania, klasa AMB ma kilka przykładów), więc wyniki mają duży rozrzut. Podajemy przedział bootstrapowy.

In [ ]:
def parse_mods(cell):
    out = []
    for item in str(cell).split(";"):
        item = item.strip()
        if not item:
            continue
        if "=" not in item:
            out.append((item, None, 1))
            continue
        phrase, typ = item.rsplit("=", 1)
        occ = 1
        if "@" in phrase:
            phrase, occ_s = phrase.rsplit("@", 1)
            occ = int(occ_s) if occ_s.strip().isdigit() else 1
        out.append((phrase.strip(), typ.strip().upper(), occ))
    return out


def token_labels(text, cell):
    toks = text.split()
    labels = ["O"] * len(toks)
    for phrase, typ, occ in parse_mods(cell):
        ptoks = [p.lower() for p in phrase.split()]
        if typ not in MOD_TYPES or not ptoks:
            continue
        starts = [i for i in range(len(toks) - len(ptoks) + 1)
                  if [t.lower() for t in toks[i:i + len(ptoks)]] == ptoks]
        if len(starts) < occ:
            continue
        s = starts[occ - 1]
        for j in range(s, s + len(ptoks)):
            labels[j] = typ
    return toks, labels


df = pd.read_csv(DATA, index_col=0, dtype=str, keep_default_na=False)
df = df.rename(columns={"sentyment": "label"}).rename_axis("id").reset_index()
df = df[df["label"].isin(SENT_LABELS)].reset_index(drop=True)
df["words"] = df["text"].str.split()
df["word_labels"] = [token_labels(t, m)[1] for t, m in zip(df["text"], df["modyfikatory"])]
if "gold" not in df:
    df["gold"] = ""

trainval, test_df = train_test_split(df, test_size=0.15, stratify=df["label"], random_state=SEED)
train_df, val_df = train_test_split(trainval, test_size=0.15 / 0.85, stratify=trainval["label"], random_state=SEED)
train_df, val_df, test_df = (d.reset_index(drop=True) for d in (train_df, val_df, test_df))

print(f"train={len(train_df)}, val={len(val_df)}, test={len(test_df)}")
display(pd.DataFrame({n: d["label"].value_counts() for n, d in
                      [("train", train_df), ("val", val_df), ("test", test_df)]}).reindex(SENT_LABELS))
tok_counts = pd.DataFrame({n: pd.Series(Counter(l for wl in d["word_labels"] for l in wl)) for n, d in
                           [("train", train_df), ("val", val_df), ("test", test_df)]}).reindex(TOK_LABELS)
print("Etykiety słów (zadanie tokenowe):")
display(tok_counts)

## Wspólna infrastruktura: modele, zbiory, trening, ocena

Jedna zestaw funkcji obsługuje obie rodziny modeli (`kind` = `"bert"` / `"gpt2"`) i oba zadania (`task` = `"seq"` / `"tok"`), więc kroki 1 i 3 różnią się tylko argumentami.

- **Tokenizer:** dla GPT-2 `add_prefix_space=True` (potrzebne przy `is_split_into_words`) oraz `pad_token = eos_token`, padding po prawej.
- **Zbiory:** dla klasyfikacji tekstu etykieta to indeks klasy; dla tokenów pierwsze podsłowo słowa dostaje etykietę, reszta i tokeny specjalne `-100`. Padding do najdłuższego w batchu robi collator (`DataCollatorWithPadding`, `DataCollatorForTokenClassification`), który **pady etykiet również wypełnia `-100`**.
- **Wagi klas:** klasy są niezbalansowane (AMB w tekście, `O` w tokenach), więc używamy łagodnych wag $\sqrt{N / (K n_k)}$ w funkcji straty (`WeightedTrainer`).
- **LoRA:** BERT `r=16`, `query`+`value`; GPT-2 `r=16`, `c_attn` z `fan_in_fan_out=True`. Głowica (`classifier` / `score`) jest uczona w całości dzięki `task_type`.
- **Wybór checkpointu:** po macro-F1 na walidacji (`load_best_model_at_end`).

In [ ]:
CLASSES = {("bert", "seq"): BertForSequenceClassification, ("bert", "tok"): BertForTokenClassification,
           ("gpt2", "seq"): GPT2ForSequenceClassification, ("gpt2", "tok"): GPT2ForTokenClassification}
LORA = {"bert": dict(r=16, lora_alpha=32, lora_dropout=0.1, target_modules=["query", "value"]),
        "gpt2": dict(r=16, lora_alpha=32, lora_dropout=0.1, target_modules=["c_attn"], fan_in_fan_out=True)}
HP = {"bert": dict(lr=5e-4, epochs=15, bs=16), "gpt2": dict(lr=3e-4, epochs=15, bs=8)}


def load_tokenizer(kind):
    kw = {"add_prefix_space": True} if kind == "gpt2" else {}
    tok = AutoTokenizer.from_pretrained(NAMES[kind], **kw)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token       # GPT-2 nie ma tokenu paddingu
    tok.padding_side = "right"              # wymagane przez GPT2ForSequenceClassification
    return tok


def build_seq_ds(tok, d):
    enc = tok(d["text"].tolist(), truncation=True, max_length=MAX_LEN)
    data = dict(enc)
    data["labels"] = [label2id[l] for l in d["label"]]
    return Dataset.from_dict(data)


def build_tok_ds(tok, d):
    enc = tok(d["words"].tolist(), is_split_into_words=True, truncation=True, max_length=MAX_LEN)
    all_labels = []
    for i, wl in enumerate(d["word_labels"]):
        prev, labs = None, []
        for wid in enc.word_ids(i):
            if wid is None:                 # token specjalny
                labs.append(-100)
            elif wid != prev:               # pierwsze podsłowo słowa
                labs.append(tag2id[wl[wid]])
            else:                           # kolejne podsłowa
                labs.append(-100)
            prev = wid
        all_labels.append(labs)
    data = dict(enc)
    data["labels"] = all_labels
    return Dataset.from_dict(data)


def build_peft(kind, task, tok):
    labels = SENT_LABELS if task == "seq" else TOK_LABELS
    kw = dict(num_labels=len(labels), id2label=dict(enumerate(labels)), label2id={l: i for i, l in enumerate(labels)})
    if kind == "gpt2":
        kw["pad_token_id"] = tok.pad_token_id      # GPT2ForSequenceClassification szuka ostatniego tokenu != pad
    model = CLASSES[(kind, task)].from_pretrained(NAMES[kind], **kw)
    cfg = LoraConfig(task_type=TaskType.SEQ_CLS if task == "seq" else TaskType.TOKEN_CLS, **LORA[kind])
    return get_peft_model(model, cfg)


def class_weights(d, task):
    if task == "seq":
        cnt = d["label"].value_counts().reindex(SENT_LABELS).fillna(0).to_numpy()
    else:
        cnt = pd.Series([l for wl in d["word_labels"] for l in wl]).value_counts().reindex(TOK_LABELS).fillna(0).to_numpy()
    cnt = np.maximum(cnt, 1)
    w = (cnt.sum() / (len(cnt) * cnt)) ** 0.5
    return torch.tensor(w / w.mean(), dtype=torch.float)


class WeightedTrainer(Trainer):
    """Trainer z ważoną entropią krzyżową; pozycje z etykietą -100 są pomijane."""

    def __init__(self, *args, class_weights=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.cw = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        w = self.cw.to(logits.device) if self.cw is not None else None
        loss = nn.functional.cross_entropy(logits.reshape(-1, logits.size(-1)).float(), labels.reshape(-1),
                                           weight=w, ignore_index=-100)
        return (loss, outputs) if return_outputs else loss


def tok_metrics(p):
    pred, y = p.predictions.argmax(-1), p.label_ids
    m = y != -100                                   # tylko pierwsze podsłowa słów
    yt, yp = y[m], pred[m]
    return {"accuracy": accuracy_score(yt, yp),
            "macro_f1": f1_score(yt, yp, average="macro", labels=[0, 1, 2, 3], zero_division=0),
            "mod_f1": f1_score(yt, yp, average="micro", labels=[1, 2, 3], zero_division=0)}


def run_training(kind, task, train_d, val_d, tag, seed=SEED, use_weights=True, epochs=None):
    set_seed(seed)
    tok = load_tokenizer(kind)
    builder = build_seq_ds if task == "seq" else build_tok_ds
    tr, va = builder(tok, train_d), builder(tok, val_d)
    model = build_peft(kind, task, tok)
    model.print_trainable_parameters()
    hp = HP[kind]
    args = TrainingArguments(
        output_dir=str(OUT / tag), learning_rate=hp["lr"], per_device_train_batch_size=hp["bs"],
        per_device_eval_batch_size=32, num_train_epochs=epochs or hp["epochs"], weight_decay=0.01,
        warmup_ratio=0.1, eval_strategy="epoch", save_strategy="epoch", save_total_limit=1,
        load_best_model_at_end=True, metric_for_best_model="macro_f1", greater_is_better=True,
        logging_strategy="epoch", report_to="none", label_names=["labels"], seed=seed,
        fp16=(DEVICE == "cuda"))
    collator = DataCollatorWithPadding(tok) if task == "seq" else DataCollatorForTokenClassification(tok)
    trainer = WeightedTrainer(model=model, args=args, train_dataset=tr, eval_dataset=va, data_collator=collator,
                              compute_metrics=seq_metrics if task == "seq" else tok_metrics,
                              processing_class=tok,
                              class_weights=class_weights(train_d, task) if use_weights else None)
    trainer.train()
    return trainer, tok


def bootstrap_ci(y, pred, n=1000, seed=SEED):
    rng = np.random.default_rng(seed)
    y, pred = np.asarray(y), np.asarray(pred)
    s = [f1_score(y[i], pred[i], average="macro") for i in (rng.integers(0, len(y), len(y)) for _ in range(n))]
    return np.percentile(s, [2.5, 97.5])


def evaluate_seq(trainer, tok, test_d, verbose=True):
    out = trainer.predict(build_seq_ds(tok, test_d))
    pred, y = out.predictions.argmax(-1), out.label_ids
    res = {"accuracy": accuracy_score(y, pred), "macro_f1": f1_score(y, pred, average="macro"),
           **{f"F1 {l}": f for l, f in zip(SENT_LABELS, f1_score(y, pred, average=None, labels=[0, 1, 2, 3], zero_division=0))}}
    if verbose:
        print(classification_report(y, pred, labels=[0, 1, 2, 3], target_names=SENT_LABELS, zero_division=0))
        display(pd.DataFrame(confusion_matrix(y, pred, labels=[0, 1, 2, 3]),
                             index=[f"prawdziwa {l}" for l in SENT_LABELS], columns=[f"przew. {l}" for l in SENT_LABELS]))
        lo, hi = bootstrap_ci(y, pred)
        print(f"macro-F1 = {res['macro_f1']:.3f}, 95% przedział bootstrapowy [{lo:.3f}, {hi:.3f}] (n={len(y)})")
    return res, pred


def evaluate_tok(trainer, tok, test_d, verbose=True):
    out = trainer.predict(build_tok_ds(tok, test_d))
    pred, y = out.predictions.argmax(-1), out.label_ids
    m = y != -100
    yt, yp = y[m], pred[m]
    res = {"accuracy": accuracy_score(yt, yp), "macro_f1": f1_score(yt, yp, average="macro", labels=[0, 1, 2, 3], zero_division=0),
           "mod_f1": f1_score(yt, yp, average="micro", labels=[1, 2, 3], zero_division=0),
           **{f"F1 {l}": f for l, f in zip(TOK_LABELS, f1_score(yt, yp, average=None, labels=[0, 1, 2, 3], zero_division=0))}}
    if verbose:
        print(classification_report(yt, yp, labels=[0, 1, 2, 3], target_names=TOK_LABELS, zero_division=0))
        display(pd.DataFrame(confusion_matrix(yt, yp, labels=[0, 1, 2, 3]),
                             index=[f"prawdziwa {l}" for l in TOK_LABELS], columns=[f"przew. {l}" for l in TOK_LABELS]))
    return res, pred, y


def show_tok_examples(pred, y, test_d, n=8):
    """Pokazuje słowa oznaczone jako modyfikator w anotacji lub w predykcji: słowo [prawda -> predykcja]."""
    shown = 0
    for i, words in enumerate(test_d["words"]):
        m = y[i] != -100
        g, p = y[i][m], pred[i][m]
        parts = [f"{w}[{TOK_LABELS[a]}->{TOK_LABELS[b]}]" for w, a, b in zip(words, g, p) if a or b]
        if parts:
            print(" ".join(parts))
            shown += 1
        if shown >= n:
            break


# wyniki wszystkich eksperymentów trzymamy w jednym słowniku
R = {}

## Zadanie 1. Strojenie HerBERT + LoRA [1.5 pkt]

### 1a. Klasyfikacja tekstu (`BertForSequenceClassification`)

Sentyment zdania (`POZ/NEG/NEU/AMB`). Trening na `train`, wybór checkpointu na `val`, ocena na `test`.

In [ ]:
trainer, tok = run_training("bert", "seq", train_df, val_df, tag="bert_seq")
res, pred = evaluate_seq(trainer, tok, test_df)
R[("bert", "seq")] = {"trainer": trainer, "model": trainer.model, "tok": tok, "res": res, "pred": pred}
trainer.model.save_pretrained(OUT / "adapter_bert_seq")
pd.DataFrame(trainer.state.log_history)[["epoch", "loss", "eval_loss", "eval_macro_f1", "eval_accuracy"]].groupby("epoch").first().round(3)

### 1b. Klasyfikacja tokenów (`BertForTokenClassification`)

Typ modyfikatora każdego słowa (`O/WZM/OSL/ODW`). Metryki liczymy na **pierwszych podsłowach słów** (pozycje z etykietą różną od `-100`). Poza macro-F1 podajemy `mod_f1`: micro-F1 tylko dla klas modyfikatorów, bo dominująca klasa `O` zawyża accuracy.

In [ ]:
trainer, tok = run_training("bert", "tok", train_df, val_df, tag="bert_tok")
res, pred, y = evaluate_tok(trainer, tok, test_df)
R[("bert", "tok")] = {"trainer": trainer, "model": trainer.model, "tok": tok, "res": res}
trainer.model.save_pretrained(OUT / "adapter_bert_tok")
print("\nPrzykłady (słowo[prawda->predykcja]):")
show_tok_examples(pred, y, test_df)

### Analiza (zadanie 1)

Uzupełnij po uruchomieniu. Pytania pomocnicze:

* **Wyniki klasyfikacji tekstu:** macro-F1 i accuracy na teście z przedziałem bootstrapowym. Które klasy są najłatwiejsze, a które najtrudniejsze? W Zadaniu 1 (anotacja) trudna była klasa AMB i granica NEU–NEG, a w części II uśrednione osadzenia zdań dawały macro-F1 ≈ 0,48. Czy kontekstowy HerBERT z LoRA to poprawia? *(uzupełnij)*
* **Wyniki klasyfikacji tokenów:** F1 dla WZM, OSL, ODW. Które modyfikatory model rozpoznaje najlepiej (hipoteza: `nie` i `bardzo` są częste i jednoznaczne, więc ODW i WZM powinny wyjść lepiej niż OSL)? *(uzupełnij)*
* **Jakość anotacji a wynik:** w Zadaniu 1 zgodność anotatorek dla modyfikatorów była niższa (kappa 0,65) niż dla sentymentu (0,79), a część anotacji w "reszcie" łamała wytyczne (np. *tylko* jako OSL). Szum etykiet ogranicza górną granicę wyniku. *(skomentuj błędy z `show_tok_examples`)*
* **Rozmiar zbioru:** test ma ok. 83 zdania, a przedział bootstrapowy jest szeroki, więc różnice rzędu kilku punktów procentowych nie są istotne. *(uzupełnij)*
* **Efektywność PEFT:** ułamek trenowanych parametrów (z `print_trainable_parameters`) i rozmiar adaptera na dysku. *(uzupełnij)*

## Zadanie 2. Wizualizacja przestrzeni wektorowej przykładów testowych [1 pkt]

Dla każdego zdania testowego wyciągamy reprezentację tekstu i rzutujemy ją t-SNE na płaszczyznę (odległość kosinusowa).

- **BERT (adapter PEFT):** reprezentacją jest wektor tokenu **`[CLS]`** (u HerBERT-a `<s>`) z ostatniej warstwy, czyli `hidden_states[-1][:, 0]`.
- Porównujemy przestrzeń **przed strojeniem** (model bazowy z nieuczonym adapterem; dzięki inicjalizacji $B = 0$ jest on identyczny z bazowym) i **po strojeniu**.
- Wykres jest interaktywny (plotly): kolor to nasza etykieta, kształt punktu mówi, czy predykcja jest poprawna (`x` = błąd), w podpowiedzi widać zdanie, etykietę PolEmo i predykcję.
- Ilościowo: współczynnik **silhouette** (kosinusowy, względem etykiet) oraz ARI / NMI dla KMeans (k=4). Wyższe wartości po strojeniu oznaczają, że przestrzeń porządkuje się według etykiet.

In [ ]:
@torch.no_grad()
def extract_embeddings(model, tok, texts, kind, batch=32):
    """Zwraca (reprezentacje tekstu, logity). BERT: wektor [CLS]; GPT-2: wektor ostatniego tokenu != padding."""
    model.eval()
    dev = next(model.parameters()).device
    embs, logits = [], []
    for i in range(0, len(texts), batch):
        enc = tok(texts[i:i + batch], truncation=True, max_length=MAX_LEN, padding=True, return_tensors="pt").to(dev)
        out = model(**enc, output_hidden_states=True)
        hs = out.hidden_states[-1]                                  # (B, T, H), ostatnia warstwa
        if kind == "bert":
            e = hs[:, 0]                                            # [CLS] / <s>
        else:
            last = enc["attention_mask"].sum(1) - 1                 # padding po prawej
            e = hs[torch.arange(hs.size(0), device=dev), last]      # ostatni token tekstu
        embs.append(e.float().cpu().numpy())
        logits.append(out.logits.float().cpu().numpy())
    return np.vstack(embs), np.vstack(logits)


def embeddings_before(kind, texts):
    tok = load_tokenizer(kind)
    m = build_peft(kind, "seq", tok).to(DEVICE)
    E, _ = extract_embeddings(m, tok, texts, kind)
    del m
    gc.collect()
    torch.cuda.empty_cache()
    return E


def tsne_2d(X, seed=SEED):
    X = normalize(X)
    perp = max(2, min(30, (len(X) - 1) / 3))
    return TSNE(n_components=2, perplexity=perp, init="pca", metric="cosine",
                random_state=seed, learning_rate="auto").fit_transform(X)


def space_scores(X, y):
    Xn = normalize(X)
    km = KMeans(n_clusters=4, n_init=10, random_state=SEED).fit(Xn)
    return {"silhouette (cos)": silhouette_score(Xn, y, metric="cosine"),
            "ARI (KMeans)": adjusted_rand_score(y, km.labels_), "NMI (KMeans)": normalized_mutual_info_score(y, km.labels_)}


def visualize_kind(kind, title):
    r = R[(kind, "seq")]
    texts, y = test_df["text"].tolist(), test_df["label"].to_numpy()
    E_before = embeddings_before(kind, texts)
    E_after, logits = extract_embeddings(r["model"], r["tok"], texts, kind)
    pred = np.array([SENT_LABELS[i] for i in logits.argmax(1)])

    parts = []
    for stage, E in [("przed strojeniem", E_before), ("po strojeniu", E_after)]:
        xy = tsne_2d(E)
        d = pd.DataFrame({"x": xy[:, 0], "y": xy[:, 1], "etap": stage, "etykieta": y,
                          "zdanie": test_df["text"].str.slice(0, 140).values, "PolEmo": test_df["gold"].values})
        d["predykcja"] = pred if stage == "po strojeniu" else "-"
        d["poprawna"] = np.where(pred == y, "tak", "nie") if stage == "po strojeniu" else "-"
        parts.append(d)
    plot_df = pd.concat(parts, ignore_index=True)
    fig = px.scatter(plot_df, x="x", y="y", color="etykieta", symbol="poprawna", facet_col="etap",
                     color_discrete_map=COLORS, symbol_map={"tak": "circle", "nie": "x", "-": "circle"},
                     category_orders={"etykieta": SENT_LABELS, "etap": ["przed strojeniem", "po strojeniu"]},
                     hover_name="zdanie", hover_data={"x": False, "y": False, "etap": False, "predykcja": True, "PolEmo": True},
                     title=title, width=1250, height=620)
    fig.update_traces(marker=dict(size=10, opacity=0.85))
    fig.update_xaxes(matches=None, showticklabels=False)
    fig.update_yaxes(matches=None, showticklabels=False)
    fig.write_html(OUT / f"tsne_testowe_{kind}.html")
    fig.show()
    scores = pd.DataFrame({"przed strojeniem": space_scores(E_before, y), "po strojeniu": space_scores(E_after, y)}).T
    display(scores.round(3))
    return scores


scores_bert = visualize_kind("bert", "HerBERT + LoRA: reprezentacja [CLS] zdań testowych (t-SNE)")

### Analiza (zadanie 2, BERT)

* **Przed strojeniem vs po strojeniu:** czy po strojeniu zdania układają się w skupiska zgodne z etykietami? Jak zmieniły się silhouette, ARI i NMI? *(uzupełnij)*
* **Które klasy się mieszają?** Hipoteza: POZ i NEG rozdzielają się najlepiej, a AMB leży pomiędzy nimi i przy NEU. Porównaj z macierzą pomyłek z zadania 1a. *(uzupełnij)*
* **Punkty oznaczone `x` (błędy):** leżą na granicach skupisk czy w "cudzych" skupiskach? Sprawdź w podpowiedziach ich treść i etykietę PolEmo. Część z nich może być sporna już na poziomie anotacji (Zadanie 1). *(uzupełnij)*
* **Porównanie z częścią II Zadania 1:** tam uśrednione osadzenia słów nie tworzyły skupisk emocjonalnych (ARI ≤ 0,08). *(uzupełnij: czy `[CLS]` po strojeniu jest wyraźnie lepszy?)*
* **Zastrzeżenie:** t-SNE zniekształca odległości globalne, a próba (ok. 83 punkty) jest mała, więc wnioskujemy o skupiskach lokalnych i podpieramy się miarami ilościowymi.

## Zadanie 3. To samo dla GPT-2 (`GPT2ForSequenceClassification`, `GPT2ForTokenClassification`) [1 pkt]

Model `sdadas/polish-gpt2-medium` + LoRA (`c_attn`, `fan_in_fan_out=True`). Różnice względem BERT-a:

- reprezentacja tekstu w wizualizacji to **ostatni token sekwencji** (nie `[CLS]`),
- `pad_token = eos_token`, padding po prawej, `pad_token_id` ustawiony w konfiguracji modelu,
- w zadaniu tokenowym tokeny **paddingu i tokeny specjalne maskujemy etykietą `-100`**: robi to funkcja `build_tok_ds` (tokeny specjalne i kolejne podsłowa) oraz `DataCollatorForTokenClassification` (padding), a strata ignoruje te pozycje.

Sprawdźmy najpierw, jak tokenizer dzieli słowa i czy maskowanie działa zgodnie z oczekiwaniem.

In [ ]:
tok_g = load_tokenizer("gpt2")
ex = train_df.iloc[0]
enc = tok_g(ex["words"], is_split_into_words=True)
ds_demo = build_tok_ds(tok_g, train_df.iloc[:2])
batch = DataCollatorForTokenClassification(tok_g)([ds_demo[0], ds_demo[1]])
print("pad_token:", tok_g.pad_token, "| pad_token_id:", tok_g.pad_token_id, "| padding_side:", tok_g.padding_side)
print("podsłowa      :", tok_g.convert_ids_to_tokens(enc["input_ids"])[:14])
print("word_ids      :", enc.word_ids()[:14])
print("etykiety batch:", batch["labels"][0][:14].tolist(), "...")
print("maska pad (-100) w krótszej sekwencji:", int((batch["labels"] == -100).sum()), "pozycji ogółem w batchu")

In [ ]:
# 3a. klasyfikacja tekstu
trainer, tok = run_training("gpt2", "seq", train_df, val_df, tag="gpt2_seq")
res, pred = evaluate_seq(trainer, tok, test_df)
R[("gpt2", "seq")] = {"trainer": trainer, "model": trainer.model, "tok": tok, "res": res, "pred": pred}
trainer.model.save_pretrained(OUT / "adapter_gpt2_seq")

In [ ]:
# 3b. klasyfikacja tokenów
trainer, tok = run_training("gpt2", "tok", train_df, val_df, tag="gpt2_tok")
res, pred, y = evaluate_tok(trainer, tok, test_df)
R[("gpt2", "tok")] = {"trainer": trainer, "model": trainer.model, "tok": tok, "res": res}
trainer.model.save_pretrained(OUT / "adapter_gpt2_tok")
print("\nPrzykłady (słowo[prawda->predykcja]):")
show_tok_examples(pred, y, test_df)

In [ ]:
# 3c. wizualizacja: reprezentacja ostatniego tokenu
scores_gpt = visualize_kind("gpt2", "GPT-2 + LoRA: reprezentacja ostatniego tokenu zdań testowych (t-SNE)")

In [ ]:
summary = pd.DataFrame({f"{NAMES[k].split('/')[-1]} / {'tekst' if t == 'seq' else 'tokeny'}": v["res"]
                        for (k, t), v in R.items()}).T
print("Zestawienie wyników na zbiorze testowym:")
display(summary.round(3))

### Analiza (zadanie 3)

* **Klasyfikacja tekstu, GPT-2 vs HerBERT:** macro-F1 i accuracy z przedziałami. Spodziewamy się, że model polski enkoderowy będzie co najmniej tak dobry jak dekoder, ale przy ok. 83 przykładach testowych różnica może być w granicach błędu. *(uzupełnij)*
* **Klasyfikacja tokenów, GPT-2 vs HerBERT:** hipoteza: GPT-2 wypada gorzej, bo maska przyczynowa pozbawia token kontekstu prawego (np. słowo modyfikowane stoi **po** modyfikatorze). Porównaj F1 dla WZM, OSL, ODW i przykłady błędów. *(uzupełnij)*
* **Wizualizacja (ostatni token vs `[CLS]`):** porównaj silhouette/ARI/NMI i układ punktów. Ostatni token zdania to zwykle kropka, której reprezentacja skupia kontekst całego zdania. *(uzupełnij)*
* **Maskowanie `-100`:** ile pozycji w batchu jest maskowanych (padding + kolejne podsłowa)? Co by się stało bez maskowania? Model uczyłby się przewidywać etykietę dla paddingu i wnętrza słów, co zawyżałoby metryki i rozmywało stratę. *(uzupełnij)*
* **Koszt:** liczba parametrów trenowanych i czas treningu GPT-2 medium vs HerBERT. *(uzupełnij)*

## Zadanie 4. Augmentacja danych tokenem `[MASK]` (BERT) [0.5 pkt]

**Procedura** dla każdego zdania treningowego:

1. wybieramy ok. 15% słów, ale **tylko** słowa etykiety `O` (nie ruszamy modyfikatorów, więc etykiety tokenów pozostają poprawne, a zdania nadają się też do zadania tokenowego), złożone z liter i dłuższe niż 2 znaki,
2. podmieniamy słowo na `<mask>` i prosimy model MLM o przewidywania; z listy top-k wybieramy **pełne słowa** (nie fragmenty podsłów), różne od oryginału, i losujemy zgodnie z prawdopodobieństwami (temperatura),
3. powtarzamy dla kolejnych wybranych słów (każda podmiana widzi poprzednie),
4. **równoważymy klasy:** mniej licznym klasom generujemy więcej wariantów (od 1 do 5 na zdanie),
5. **filtrujemy** klasyfikatorem z kroku 1a: zachowujemy wariant, jeśli klasyfikator przypisuje mu oryginalną etykietę z pewnością $\ge$ `MIN_CONF` (ochrona przed antonimami, które MLM chętnie podpowiada).

> Sprawdź w logu ładowania modelu, czy nie ma ostrzeżenia, że wagi głowicy MLM (`cls.predictions...`) zostały zainicjowane losowo. Wtedy `allegro/herbert-base-cased` nie ma głowicy MLM i trzeba ustawić `MLM_NAME` na model z głowicą (np. `allegro/herbert-large-cased` lub inny polski BERT).

In [ ]:
mlm_tok = AutoTokenizer.from_pretrained(MLM_NAME)
mlm = AutoModelForMaskedLM.from_pretrained(MLM_NAME).to(DEVICE).eval()
USES_W = any(t.endswith("</w>") for t in list(mlm_tok.get_vocab())[:5000])    # BPE HerBERT-a oznacza koniec słowa


def full_word(token_str):
    if USES_W:
        return token_str.endswith("</w>")
    return not token_str.startswith("##")


def softmax_np(z):
    z = np.asarray(z, dtype=float)
    z = z - z.max()
    e = np.exp(z)
    return e / e.sum()


@torch.no_grad()
def mlm_augment(words, wlabels, rng, p_mask=0.15, top_k=10, temperature=1.0):
    cand_idx = [i for i, (w, l) in enumerate(zip(words, wlabels)) if l == "O" and re.fullmatch(r"[^\W\d_]{3,}", w)]
    n = max(1, round(p_mask * len(words)))
    new = list(words)
    for i in rng.sample(cand_idx, min(n, len(cand_idx))):
        masked = new.copy()
        masked[i] = mlm_tok.mask_token
        enc = mlm_tok(" ".join(masked), return_tensors="pt", truncation=True, max_length=MAX_LEN).to(DEVICE)
        pos = (enc["input_ids"][0] == mlm_tok.mask_token_id).nonzero()
        if len(pos) == 0:
            continue
        logits = mlm(**enc).logits[0, pos[0, 0]]
        cands = []
        for t in torch.topk(logits, top_k * 8).indices.tolist():
            ts = mlm_tok.convert_ids_to_tokens(t)
            if not full_word(ts):
                continue
            w = ts.replace("</w>", "").strip()
            if w.isalpha() and w.lower() != new[i].lower():
                cands.append((w, logits[t].item()))
            if len(cands) == top_k:
                break
        if not cands:
            continue
        pick = rng.choices([w for w, _ in cands], weights=softmax_np([s for _, s in cands]) ** (1 / temperature))[0]
        new[i] = pick.capitalize() if words[i][:1].isupper() else pick
    return new


rng = random.Random(SEED)
for _, r_ in train_df.head(4).iterrows():
    print("ORYG:", r_["text"])
    print("MLM :", " ".join(mlm_augment(r_["words"], r_["word_labels"], rng)), "\n")

In [ ]:
N_MAX_VARIANTS, P_MASK, TOP_K_MLM, T_MLM, MIN_CONF = 5, 0.15, 10, 1.0, 0.6


def predict_labels(texts, kind="bert"):
    """Klasyfikator z kroku 1a/3a: etykieta, pewność i rozkład."""
    r = R[(kind, "seq")]
    _, logits = extract_embeddings(r["model"], r["tok"], texts, kind)
    p = np.exp(logits - logits.max(1, keepdims=True))
    p /= p.sum(1, keepdims=True)
    return np.array([SENT_LABELS[i] for i in p.argmax(1)]), p.max(1), p


rng = random.Random(SEED)
counts = train_df["label"].value_counts()
variants = {l: int(np.clip(round(counts.max() / counts[l]), 1, N_MAX_VARIANTS)) for l in SENT_LABELS}
print("wariantów na zdanie:", variants)

rows = []
for _, r_ in train_df.iterrows():
    seen = {r_["text"]}
    for _ in range(variants[r_["label"]]):
        w = mlm_augment(r_["words"], r_["word_labels"], rng, P_MASK, TOP_K_MLM, T_MLM)
        t = " ".join(w)
        if t not in seen:
            seen.add(t)
            rows.append({"id": f"mlm_{len(rows)}", "text": t, "label": r_["label"], "words": w,
                         "word_labels": r_["word_labels"], "modyfikatory": r_["modyfikatory"], "gold": r_["gold"], "src": r_["text"]})
aug_mlm_raw = pd.DataFrame(rows)

pred_l, conf, _ = predict_labels(aug_mlm_raw["text"].tolist())
aug_mlm_raw["pred"], aug_mlm_raw["conf"] = pred_l, conf
aug_mlm_raw["keep"] = (aug_mlm_raw["pred"] == aug_mlm_raw["label"]) & (aug_mlm_raw["conf"] >= MIN_CONF)
aug_mlm = aug_mlm_raw[aug_mlm_raw["keep"]].reset_index(drop=True)

print(f"wygenerowano {len(aug_mlm_raw)}, po filtrze {len(aug_mlm)} ({len(aug_mlm) / max(len(aug_mlm_raw), 1):.0%})")
display(pd.DataFrame({"przed filtrem": aug_mlm_raw["label"].value_counts(), "po filtrze": aug_mlm["label"].value_counts(),
                      "train oryginalny": counts}).reindex(SENT_LABELS))
print("\nPrzykłady odrzucone (zmiana etykiety wg klasyfikatora):")
display(aug_mlm_raw[~aug_mlm_raw["keep"]][["src", "text", "label", "pred", "conf"]].head(6).round(2))
print("Przykłady zachowane:")
display(aug_mlm[["src", "text", "label", "conf"]].head(6).round(2))
aug_mlm.to_csv(OUT / "augmentacja_mlm.csv", index=False, encoding="utf-8-sig")

### Analiza (zadanie 4)

* **Odsetek odrzuconych wariantów** (filtr klasyfikatorem) i charakter odrzuceń: czy MLM podstawia antonimy (*dobry* → *zły*) albo słowa zmieniające sens? Odnieś do wniosków z części II Zadania 1 (antonimy mają podobne konteksty). *(uzupełnij)*
* **Jakość składniowa i fleksyjna:** czy podmienione słowa pasują gramatycznie (przypadek, rodzaj)? Polska fleksja utrudnia pojedyncze podmiany. *(uzupełnij)*
* **Wpływ parametrów** (`P_MASK`, `TOP_K_MLM`, `T_MLM`, `MIN_CONF`): kompromis między różnorodnością a zachowaniem etykiety. *(uzupełnij: możesz powtórzyć eksperyment dla innych wartości)*
* **Ograniczenia filtra:** klasyfikator trenowany na tych samych danych odrzuci właśnie trudne, informacyjne warianty i zachowa łatwe (ryzyko, że augmentacja niewiele wnosi). *(skomentuj wynik z zadania 6)*

## Zadanie 5. Augmentacja generatywnym GPT-2 [1 pkt]

**Krok 5a: generator warunkowany klasą.** Dostrajamy (LoRA, `GPT2LMHeadModel`) `polish-gpt2-medium` na zdaniach treningowych w formacie

```
Ocena pozytywna: <zdanie>\n
```

(`pozytywna / negatywna / neutralna / mieszana`), a znak nowej linii pełni rolę końca zdania (nie używamy `eos`, bo jest tokenem paddingu i `DataCollatorForLanguageModeling` zamaskowałby go etykietą `-100`). Generator widzi **wyłącznie zbiór treningowy**.

**Krok 5b: eksperyment z hiperparametrami generacji.** Dla każdej klasy bierzemy prompty `Ocena <klasa>: <2 pierwsze słowa realnego zdania>` i generujemy kontynuację w kilku konfiguracjach (greedy, beam, różne temperatury, top-k, top-p). Mierzymy:

- **distinct-1/2:** odsetek unikalnych unigramów/bigramów (różnorodność),
- **duplikaty:** odsetek powtórzonych zdań,
- **nowość:** odsetek zdań, których nie ma dosłownie w zbiorze treningowym,
- **zgodność etykiety:** odsetek zdań, którym klasyfikator z kroku 1a przypisuje etykietę z promptu.

**Krok 5c:** wybieramy najlepsze konfiguracje, generujemy zdania, filtrujemy i deduplikujemy.

In [ ]:
LABEL_WORD = {"POZ": "pozytywna", "NEG": "negatywna", "NEU": "neutralna", "AMB": "mieszana"}
lm_tok = AutoTokenizer.from_pretrained(GPT_NAME)
if lm_tok.pad_token is None:
    lm_tok.pad_token = lm_tok.eos_token
NEWLINE_ID = lm_tok("\n", add_special_tokens=False)["input_ids"][-1]


def fmt(label, text):
    return f"Ocena {LABEL_WORD[label]}: {text}\n"


lm_ds = Dataset.from_dict({"text": [fmt(l, t) for l, t in zip(train_df["label"], train_df["text"])]})
lm_ds = lm_ds.map(lambda b: lm_tok(b["text"], truncation=True, max_length=MAX_LEN), batched=True, remove_columns=["text"])

set_seed(SEED)
lm_base = GPT2LMHeadModel.from_pretrained(GPT_NAME)
lm = get_peft_model(lm_base, LoraConfig(task_type=TaskType.CAUSAL_LM, r=16, lora_alpha=32, lora_dropout=0.1,
                                        target_modules=["c_attn"], fan_in_fan_out=True))
lm.print_trainable_parameters()
lm_trainer = Trainer(
    model=lm,
    args=TrainingArguments(output_dir=str(OUT / "gpt2_generator"), learning_rate=3e-4, per_device_train_batch_size=8,
                           num_train_epochs=5, weight_decay=0.01, warmup_ratio=0.1, logging_strategy="epoch",
                           save_strategy="no", report_to="none", seed=SEED, fp16=(DEVICE == "cuda")),
    train_dataset=lm_ds, data_collator=DataCollatorForLanguageModeling(lm_tok, mlm=False), processing_class=lm_tok)
lm_trainer.train()
lm = lm_trainer.model.eval()
lm.save_pretrained(OUT / "adapter_gpt2_generator")

In [ ]:
GEN_CONFIGS = {
    "greedy": dict(do_sample=False),
    "beam=4, no-repeat-3": dict(do_sample=False, num_beams=4, no_repeat_ngram_size=3),
    "T=0.7": dict(do_sample=True, temperature=0.7, top_k=0),
    "T=1.0, top-k=50": dict(do_sample=True, temperature=1.0, top_k=50),
    "T=1.0, top-p=0.9": dict(do_sample=True, temperature=1.0, top_p=0.9, top_k=0),
    "T=1.3, top-p=0.95": dict(do_sample=True, temperature=1.3, top_p=0.95, top_k=0),
}


@torch.no_grad()
def gpt_generate(prompts, cfg, max_new_tokens=40, batch=16):
    lm_tok.padding_side = "left"                     # generacja wsadowa wymaga paddingu po lewej
    conts = []
    for i in range(0, len(prompts), batch):
        enc = lm_tok(prompts[i:i + batch], return_tensors="pt", padding=True).to(DEVICE)
        gen = lm.generate(**enc, max_new_tokens=max_new_tokens, pad_token_id=lm_tok.pad_token_id,
                          eos_token_id=NEWLINE_ID, **cfg)
        conts += lm_tok.batch_decode(gen[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)
    return conts


def make_prompts(label, n, rng, seed_words=2):
    pool = train_df[train_df["label"] == label]
    rows = [pool.iloc[rng.randrange(len(pool))] for _ in range(n)]
    seeds = [" ".join(r_["words"][:seed_words]) for r_ in rows]
    return [f"Ocena {LABEL_WORD[label]}: {s}" for s in seeds], seeds


def generate_for_label(label, n, cfg, rng, seed_words=2):
    prompts, seeds = make_prompts(label, n, rng, seed_words)
    conts = gpt_generate(prompts, cfg)
    return [" ".join((s + c).split("\n")[0].split()) for s, c in zip(seeds, conts)]


def distinct_n(texts, n):
    grams = []
    for t in texts:
        w = t.split()
        grams += [tuple(w[i:i + n]) for i in range(len(w) - n + 1)]
    return len(set(grams)) / max(len(grams), 1)


N_PER_CLASS_EXP = 20
train_texts = set(train_df["text"])
exp_rows, samples = [], {}
for name, cfg in GEN_CONFIGS.items():
    rng = random.Random(SEED)
    gen = [(l, t) for l in SENT_LABELS for t in generate_for_label(l, N_PER_CLASS_EXP, cfg, rng)]
    texts = [t for _, t in gen]
    pred_l, conf, _ = predict_labels(texts)
    exp_rows.append({"konfiguracja": name, "liczba": len(texts), "śr. długość (słowa)": np.mean([len(t.split()) for t in texts]),
                     "distinct-1": distinct_n(texts, 1), "distinct-2": distinct_n(texts, 2),
                     "duplikaty": 1 - len(set(texts)) / len(texts), "nowe (nie w train)": np.mean([t not in train_texts for t in texts]),
                     "zgodność etykiety": np.mean([a == l for a, (l, _) in zip(pred_l, gen)])})
    samples[name] = gen[:2] + gen[-2:]
exp_df = pd.DataFrame(exp_rows).set_index("konfiguracja")
display(exp_df.round(3))
for name, s in samples.items():
    print(f"\n--- {name}")
    for l, t in s:
        print(f"[{l}] {t}")

### Analiza (zadanie 5, eksperyment z hiperparametrami)

* **Greedy i beam:** zwykle dają mało różnorodne i powtarzalne zdania (niskie distinct-n, dużo duplikatów), więc słabo nadają się do augmentacji. *(zweryfikuj)*
* **Temperatura:** niska $T$ = wysoka zgodność etykiety, ale małe distinct-n; wysoka $T$ = różnorodność, lecz więcej nonsensów i spadek zgodności etykiety. *(uzupełnij na podstawie tabeli)*
* **Top-k vs top-p:** top-p dopasowuje liczbę kandydatów do pewności modelu, więc zwykle jest stabilniejszy przy wyższej temperaturze. *(uzupełnij)*
* **Jakość językowa:** oceń ręcznie kilka próbek (fleksja, składnia, spójność z etykietą). Zdania dla klasy AMB są najtrudniejsze do wygenerowania. *(uzupełnij)*
* **Wybór konfiguracji do augmentacji:** kompromis zgodność etykiety vs różnorodność. *(wskaż i uzasadnij w `AUG_CFGS` poniżej)*

In [ ]:
AUG_CFGS = ["T=1.0, top-p=0.9", "T=1.0, top-k=50"]      # <- wybierz na podstawie tabeli wyżej
TARGET_PER_CLASS = int(train_df["label"].value_counts().max() * 1.0)   # docelowo tyle zdań w klasie (oryginał + syntetyczne)
MIN_CONF_GPT = 0.6

rng = random.Random(SEED + 1)
cands = []
for l in SENT_LABELS:
    need = max(TARGET_PER_CLASS - int((train_df["label"] == l).sum()), 0)
    n_gen = int(need * 2.5) + 5            # nadmiar na odrzuty filtra i duplikaty
    for k, name in enumerate(AUG_CFGS):
        for t in generate_for_label(l, n_gen // len(AUG_CFGS) + 1, GEN_CONFIGS[name], rng):
            cands.append({"text": t, "label": l, "cfg": name})
cands = pd.DataFrame(cands).drop_duplicates("text")
cands = cands[~cands["text"].isin(train_texts) & cands["text"].str.split().str.len().between(4, 40)].reset_index(drop=True)

pred_l, conf, _ = predict_labels(cands["text"].tolist())
cands["pred"], cands["conf"] = pred_l, conf
cands["keep"] = (cands["pred"] == cands["label"]) & (cands["conf"] >= MIN_CONF_GPT)

kept = []
for l in SENT_LABELS:
    need = max(TARGET_PER_CLASS - int((train_df["label"] == l).sum()), 0)
    kept.append(cands[(cands["label"] == l) & cands["keep"]].head(need))
aug_gpt = pd.concat(kept).reset_index(drop=True)
aug_gpt["id"] = [f"gpt_{i}" for i in range(len(aug_gpt))]
aug_gpt["words"] = aug_gpt["text"].str.split()
aug_gpt["gold"] = ""

print(f"kandydaci po czyszczeniu: {len(cands)}, zachowane po filtrze: {int(cands['keep'].sum())}, użyte: {len(aug_gpt)}")
display(pd.DataFrame({"kandydaci": cands["label"].value_counts(), "użyte": aug_gpt["label"].value_counts(),
                      "train oryginalny": train_df["label"].value_counts()}).reindex(SENT_LABELS))
display(aug_gpt[["label", "text", "cfg", "conf"]].sample(min(10, len(aug_gpt)), random_state=SEED).round(2))
aug_gpt.drop(columns=["words"]).to_csv(OUT / "augmentacja_gpt2.csv", index=False, encoding="utf-8-sig")

> Zdania z GPT-2 nie mają anotacji modyfikatorów, więc używamy ich tylko do **klasyfikacji tekstu**. (Opcjonalnie można je zaanotować tokenowo modelem z kroku 1b jako etykietami "silver".)

## Zadanie 6. Czy augmentacja pomaga? Porównanie na niezmienionym zbiorze testowym

Trenujemy **jeden model** (HerBERT + LoRA, klasyfikacja tekstu) na czterech wariantach zbioru treningowego i oceniamy każdy na tym samym, **nieaugmentowanym** zbiorze walidacyjnym i testowym:

| Wariant | Zbiór treningowy |
|---|---|
| baseline | oryginalny `train` |
| + MLM | `train` + zdania z zadania 4 |
| + GPT-2 | `train` + zdania z zadania 5 |
| + oba | `train` + zdania z zadań 4 i 5 |

Wszystko powtarzamy dla kilku ziaren (`FINAL_SEEDS`), bo pojedynczy przebieg na małym zbiorze jest zbyt losowy. Raportujemy średnią i odchylenie standardowe.

In [ ]:
FINAL_SEEDS = [42, 43, 44]
cols = ["text", "label", "words", "word_labels"]
aug_gpt_ = aug_gpt.assign(word_labels=[["O"] * len(w) for w in aug_gpt["words"]])[cols]
variants_df = {
    "baseline": train_df[cols],
    "+ MLM": pd.concat([train_df[cols], aug_mlm[cols]], ignore_index=True),
    "+ GPT-2": pd.concat([train_df[cols], aug_gpt_], ignore_index=True),
    "+ MLM + GPT-2": pd.concat([train_df[cols], aug_mlm[cols], aug_gpt_], ignore_index=True),
}
print({k: len(v) for k, v in variants_df.items()})

rows = []
for vname, tr_df in variants_df.items():
    for s in FINAL_SEEDS:
        t, tk = run_training("bert", "seq", tr_df, val_df, tag=f"final_{vname}_{s}".replace(" ", "").replace("+", "p"), seed=s)
        r_, _ = evaluate_seq(t, tk, test_df, verbose=False)
        rows.append({"wariant": vname, "seed": s, **r_})
        del t
        gc.collect()
        torch.cuda.empty_cache()
final = pd.DataFrame(rows)
agg = final.groupby("wariant", sort=False)[["macro_f1", "accuracy", "F1 POZ", "F1 NEG", "F1 NEU", "F1 AMB"]].agg(["mean", "std"]).round(3)
display(agg)

### Analiza (zadanie 6) i podsumowanie

Uzupełnij po uruchomieniu:

* **Czy augmentacja poprawiła macro-F1 względem baseline'u?** Porównaj średnie z odchyleniami. Jeśli różnica mieści się w jednym odchyleniu standardowym (a test ma ok. 83 zdania), nie możemy twierdzić, że augmentacja pomaga. *(uzupełnij)*
* **Klasa AMB:** to ona powinna zyskać najwięcej (augmentacja ją wyrównuje). Czy `F1 AMB` wzrosło? *(uzupełnij)*
* **MLM vs GPT-2:** która metoda daje więcej wartościowych zdań? MLM zachowuje strukturę oryginału (mała różnorodność, mało szumu), GPT-2 daje nowe zdania (większa różnorodność, większy szum). *(uzupełnij)*
* **Wyciek:** augmentowaliśmy tylko `train`, a generator i filtr widziały wyłącznie `train`, więc wyniki na `val`/`test` są uczciwe. *(potwierdź)*
* **Podsumowanie całego ćwiczenia:**
    * adapter LoRA trenuje ułamek parametrów, a daje wyniki na poziomie pełnego strojenia (por. tutorial w części C) *(uzupełnij liczby)*,
    * HerBERT vs GPT-2: enkoder z dwustronnym kontekstem vs dekoder z lewostronnym; różnica widoczna głównie w zadaniu tokenowym *(uzupełnij)*,
    * reprezentacja `[CLS]` / ostatniego tokenu po strojeniu tworzy skupiska zgodne z etykietami, w przeciwieństwie do uśrednionych osadzeń słów z Zadania 1 *(uzupełnij)*,
    * augmentacja ma sens tylko z filtrem zachowania etykiety i z oceną na nietkniętym zbiorze testowym *(uzupełnij)*.